# 05 — Pull Fragility & Humanitarian Data

This notebook pulls **Domain 3 — Fragility/Humanitarian** predictor features.

| Dataset | Variables | Access |
|---|---|---|
| **Fragile States Index (FSI)** | 12 sub-scores + total (0–120) | Fund for Peace direct CSV download |
| **UNHCR Refugee Statistics** | Refugees hosted, IDPs, refugee outflows | UNHCR Refugee Data Finder API |
| **UNDP Human Development Index** | HDI composite + component indices | UNDP data portal direct download |

## What this notebook does
1. Downloads all available FSI annual editions and stacks into a long panel
2. Pulls UNHCR refugee/IDP counts via the UNHCR API
3. Downloads the UNDP HDI dataset
4. Writes each to ADLS as parquet

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

## Notes
- FSI sub-scores are the 12 indicators: Security Apparatus (SEC), Factionalized Elites (ELI),
  Group Grievance (GG), Economy (ECO), Economic Inequality (EQ), Human Flight (HF),
  State Legitimacy (SL), Public Services (PS), Human Rights (HR),
  Demographic Pressures (DP), Refugees & IDPs (REF), External Intervention (EX)
- UNHCR data is **also** available in WDI but the UNHCR Refugee Data Finder API
  provides more granular refugee-type breakdowns

In [ ]:
import os
import io
import requests
import pandas as pd
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# Fund for Peace publishes FSI as one CSV per year at a stable URL pattern
# The download URL requires an HTTP GET with a specific query string
FSI_BASE_URL = "https://fragilestatesindex.org/wp-content/uploads/"

# UNHCR Refugee Data Finder API v1
UNHCR_API_BASE = "https://api.unhcr.org/population/v1"

# UNDP HDI — statistical data service
UNDP_HDI_URL = (
    "https://hdr.undp.org/sites/default/files/2023-24_HDR/"
    "HDR23-24_Statistical_Annex_HDI_Table.xlsx"
)

print(f"Run date: {RUN_DATE}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## 1. Fragile States Index

Fund for Peace publishes annual FSI spreadsheets. The URL pattern changed across years;
we try a list of known URLs and fall back to a direct download from the data archive.

If the automated download fails, manually download all annual CSVs from
https://fragilestatesindex.org/excel/ and place them in a local `fsi_files/` directory,
then re-run from the **Parse FSI files** cell.

In [ ]:
# Known FSI annual download URLs (update if Fund for Peace changes their site)
FSI_YEAR_URLS = {
    2023: "https://fragilestatesindex.org/wp-content/uploads/2023/06/FSI-2023-Download.xlsx",
    2022: "https://fragilestatesindex.org/wp-content/uploads/2022/07/FSI-2022-Download.xlsx",
    2021: "https://fragilestatesindex.org/wp-content/uploads/2021/05/FSI-2021-Download.xlsx",
    2020: "https://fragilestatesindex.org/wp-content/uploads/2020/05/fsi-2020.xlsx",
    2019: "https://fragilestatesindex.org/wp-content/uploads/2019/04/9511904_FSIDataPackage2019.xlsx",
    2018: "https://fragilestatesindex.org/wp-content/uploads/2018/04/fsi-2018.xlsx",
    2017: "https://fragilestatesindex.org/wp-content/uploads/2017/06/FSI-2017.xlsx",
    2016: "https://fragilestatesindex.org/wp-content/uploads/2016/07/FSI_2016_Data.xlsx",
    2015: "https://fragilestatesindex.org/wp-content/uploads/2015/06/FSI2015-Data.xlsx",
    2014: "https://fragilestatesindex.org/wp-content/uploads/2014/06/FSI2014-Data.xlsx",
    2013: "https://fragilestatesindex.org/wp-content/uploads/2013/06/FSI2013-Data.xlsx",
}

# Standard FSI column order across most editions
FSI_SCORE_COLS = [
    "fsi_total", "sec", "eli", "gg", "eco", "eq", "hf",
    "sl", "ps", "hr", "dp", "ref", "ex",
]

FSI_SCORE_LABELS = {
    "fsi_total": "fsi_total_score",
    "sec": "fsi_security_apparatus",
    "eli": "fsi_factionalized_elites",
    "gg":  "fsi_group_grievance",
    "eco": "fsi_economy",
    "eq":  "fsi_economic_inequality",
    "hf":  "fsi_human_flight",
    "sl":  "fsi_state_legitimacy",
    "ps":  "fsi_public_services",
    "hr":  "fsi_human_rights",
    "dp":  "fsi_demographic_pressures",
    "ref": "fsi_refugees_idps",
    "ex":  "fsi_external_intervention",
}

In [ ]:
def parse_fsi_excel(content: bytes, year: int) -> pd.DataFrame | None:
    """Parse one FSI Excel file; returns a tidy long-format DataFrame."""
    try:
        df = pd.read_excel(io.BytesIO(content), sheet_name=0)
    except Exception as e:
        print(f"    Could not parse {year}: {e}")
        return None

    # Normalise column names
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]

    # Identify country column (varies: 'country', 'country_name')
    country_col = next((c for c in df.columns if "country" in c), None)
    if country_col is None:
        print(f"    No country column found in {year}")
        return None

    df = df.rename(columns={country_col: "country_name"})
    df["year"] = year

    # Identify the total score column
    total_col = next(
        (c for c in df.columns if "total" in c or "score" in c and c != "country_name"),
        None,
    )
    if total_col:
        df = df.rename(columns={total_col: "fsi_total"})

    return df[["country_name", "year"] + [c for c in FSI_SCORE_LABELS if c in df.columns]]


fsi_frames = []
for year, url in FSI_YEAR_URLS.items():
    print(f"  FSI {year}: {url}")
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        df_year = parse_fsi_excel(resp.content, year)
        if df_year is not None and len(df_year) > 0:
            fsi_frames.append(df_year)
            print(f"    OK — {len(df_year)} countries")
    except Exception as e:
        print(f"    FAILED: {e}")

if fsi_frames:
    df_fsi = pd.concat(fsi_frames, ignore_index=True)
    df_fsi.rename(columns=FSI_SCORE_LABELS, inplace=True)
    print(f"\nFSI stacked panel: {df_fsi.shape} | years {df_fsi['year'].min()}–{df_fsi['year'].max()}")
else:
    print("WARNING: no FSI data downloaded — check URLs or download manually")
    df_fsi = pd.DataFrame()

In [ ]:
if not df_fsi.empty:
    write_parquet(df_fsi, f"raw/fsi/{RUN_DATE}/fsi_panel.parquet")

## 2. UNHCR Refugee Statistics

We pull three tables from the UNHCR Refugee Data Finder API:
- `asylum-seekers` — persons seeking international protection
- `refugees` — persons under UNHCR mandate (hosted per country of asylum)
- `idps` — internally displaced persons

In [ ]:
def pull_unhcr_endpoint(endpoint: str, limit: int = 100_000) -> pd.DataFrame:
    """Pull a full UNHCR population statistics endpoint."""
    url = f"{UNHCR_API_BASE}/{endpoint}/"
    params = {"limit": limit, "yearFrom": 1995}
    resp = requests.get(url, params=params, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    return pd.DataFrame(data.get("items", []))

print("Pulling UNHCR refugees...")
df_refugees = pull_unhcr_endpoint("refugees")
print(f"  Refugees : {len(df_refugees):,} rows")

print("Pulling UNHCR IDPs...")
df_idps = pull_unhcr_endpoint("idps")
print(f"  IDPs     : {len(df_idps):,} rows")

print("Pulling UNHCR asylum seekers...")
df_asylum = pull_unhcr_endpoint("asylum-seekers")
print(f"  Asylum   : {len(df_asylum):,} rows")

In [ ]:
# Tag each table with its type before writing
for label, df in [("refugees", df_refugees), ("idps", df_idps), ("asylum", df_asylum)]:
    if not df.empty:
        df.columns = [c.lower().strip() for c in df.columns]
        write_parquet(df, f"raw/unhcr/{RUN_DATE}/unhcr_{label}.parquet")

## 3. UNDP Human Development Index

In [ ]:
print(f"Downloading UNDP HDI from {UNDP_HDI_URL} ...")
resp = requests.get(UNDP_HDI_URL, timeout=60)
resp.raise_for_status()

# HDI statistical annex is a multi-sheet Excel; the HDI table is typically sheet 0 or 1
try:
    df_hdi_raw = pd.read_excel(io.BytesIO(resp.content), sheet_name=0, header=4)
except Exception:
    df_hdi_raw = pd.read_excel(io.BytesIO(resp.content), sheet_name=1, header=4)

print(f"Raw HDI shape: {df_hdi_raw.shape}")
df_hdi_raw.head(3)

In [ ]:
# The UNDP annex is a wide table (country × years as columns); melt to long format
df_hdi = df_hdi_raw.copy()
df_hdi.columns = [str(c).strip() for c in df_hdi.columns]

# Drop completely empty rows and columns
df_hdi.dropna(how="all", inplace=True)
df_hdi.dropna(axis=1, how="all", inplace=True)

# Identify country column
country_col = next(
    (c for c in df_hdi.columns if "country" in c.lower() or "name" in c.lower()),
    df_hdi.columns[1],  # fallback: second column in UNDP annex layout
)

# Year columns are numeric strings in UNDP format
year_cols = [c for c in df_hdi.columns if str(c).isdigit() and 1990 <= int(c) <= 2030]

if year_cols:
    df_hdi_long = df_hdi[[country_col] + year_cols].melt(
        id_vars=[country_col], var_name="year", value_name="hdi"
    )
    df_hdi_long.rename(columns={country_col: "country_name"}, inplace=True)
    df_hdi_long["year"] = df_hdi_long["year"].astype(int)
    df_hdi_long["hdi"]  = pd.to_numeric(df_hdi_long["hdi"], errors="coerce")
    df_hdi_long.dropna(subset=["hdi"], inplace=True)
    print(f"HDI long panel: {df_hdi_long.shape}")
    write_parquet(df_hdi_long, f"raw/undp_hdi/{RUN_DATE}/undp_hdi_panel.parquet")
else:
    print("Could not identify year columns; writing raw table instead")
    write_parquet(df_hdi, f"raw/undp_hdi/{RUN_DATE}/undp_hdi_raw.parquet")
    df_hdi_long = df_hdi

## Summary

In [ ]:
print("=" * 55)
print("Fragility & humanitarian pull complete")
print("=" * 55)
if not df_fsi.empty:
    print(f"  FSI      : {len(df_fsi):,} country-years | {df_fsi['year'].min()}–{df_fsi['year'].max()}")
print(f"  UNHCR refugees : {len(df_refugees):,} rows")
print(f"  UNHCR IDPs     : {len(df_idps):,} rows")
print(f"  UNHCR asylum   : {len(df_asylum):,} rows")
print(f"  UNDP HDI       : {len(df_hdi_long):,} rows")
print()
print("ADLS paths written:")
print(f"  raw/fsi/{RUN_DATE}/fsi_panel.parquet")
print(f"  raw/unhcr/{RUN_DATE}/unhcr_refugees.parquet")
print(f"  raw/unhcr/{RUN_DATE}/unhcr_idps.parquet")
print(f"  raw/unhcr/{RUN_DATE}/unhcr_asylum.parquet")
print(f"  raw/undp_hdi/{RUN_DATE}/undp_hdi_panel.parquet")